In [1]:
# Traemos la tabla 'df' ya limpia desde el notebook de Feature Engineering
%run "feature-enginnering.ipynb"

# Vemos que sí se haya cargado correctamente
print("¡Datos cargados con éxito! Tamaño de la tabla:", df.shape)

===Loaded preprocessing modules: LowDoc, RevLineCr, accept, approvalDate, approvalFY, balanceGross, base_cleaning, city_bank, createJob, disbursementDate, disbursementGross, example, franchise_code, newExists, noemp, retainedJob, urban_rural
Current df features:
1. City_0
2. City_1
3. City_2
4. City_3
5. City_4
6. City_5
7. City_6
8. City_7
9. City_8
10. City_9
11. City_10
12. Bank_0
13. Bank_1
14. Bank_2
15. Bank_3
16. Bank_4
17. Bank_5
18. Bank_6
19. Bank_7
20. Bank_8
21. ApprovalDate
22. ApprovalFY
23. NewExist
24. CreateJob
25. RetainedJob
26. FranchiseCode
27. UrbanRural
28. RevLineCr
29. LowDoc
30. DisbursementDate
31. DisbursementGross
32. BalanceGross
33. Accept
34. IsLocalBank
35. NoEmp_Log
NewExist option used: A
Rows before: 20,768
Rows after: 20,768
CreateJob option used: A
Rows before: 20,768
Rows after: 20,768
RetainedJob option used: A
Rows before: 20,768
Rows after: 20,768
ApprovalDate option used: A
Rows before: 20,768
Rows after: 20,768

Opción A seleccionada: Se crea

,approvalyear_normalized,approvalmonth_normalized
0,0.590909,0.636364
1,0.840909,1.000000
2,0.590909,0.363636
3,0.909091,0.909091
4,0.840909,0.363636
5,0.750000,0.181818
6,0.795455,0.818182
7,0.522727,0.090909
8,0.659091,0.363636
9,0.772727,0.181818


ApprovalFY option used: A
Rows before: 20,768
Rows after: 20,768

Opción A seleccionada: La columna 'ApprovalFY' fue ELIMINADA del dataset.


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized,retainedjob_normalized,approvalyear_normalized,approvalmonth_normalized
0,0,0,0,0,0,0,0,0,0,0,...,$0.00,0,1,3.367296,0,0,0.0,0.0,0.590909,0.636364
1,0,0,0,0,0,0,0,0,0,1,...,$0.00,1,1,0.693147,1,0,0.000114,0.000114,0.840909,1.000000
2,0,0,0,0,0,0,0,0,0,1,...,$0.00,1,1,1.945910,1,0,0.0,0.0,0.590909,0.363636


Opción de FranchiseCode utilizada: binary
Filas antes: 20,768
Filas después: 20,768


...

Rows before: 20,768
RevLineCr option used: C
Rows after: 20,768
Rows before: 20,768
LowDoc option used: C
Rows before: 20,768
Rows after: 20,768
DisbursementGross option used: B
Rows before: 20,768
Rows after: 20,768


,DisbursementGross
0,1.469113
1,-0.572057
2,-0.59124
3,-0.395861
4,-0.48467
5,-0.243111
6,-0.48467
7,-0.129436
8,2.890045
9,-0.542573


Total features: 40


,Feature,Type
0,City_0,int64
1,City_1,int64
2,City_2,int64
3,City_3,int64
4,City_4,int64
5,City_5,int64
6,City_6,int64
7,City_7,int64
8,City_8,int64
9,City_9,int64


¡Datos cargados con éxito! Tamaño de la tabla: (20768, 40)


## Fase 4: Variantes de Boosting (Gradient Boosting & XGBoost)
**Responsable:** Ivan Angeles - Tree Team

En esta sección entrenaremos las variantes de Boosting. A diferencia de un bosque aleatorio que crea árboles independientes, el Boosting crea árboles de forma secuencial, donde cada nuevo árbol corrige los errores del anterior.

### Experimentos (Hyperparameter Tuning)
Tal como requiere la Fase 4, afinaremos (tune) los siguientes parámetros para ambos algoritmos usando Validación Cruzada (CV=5):
* **Estimators (`n_estimators`):** Número de árboles en la secuencia.
* **Depth (`max_depth`):** Profundidad máxima de cada árbol.
* **Subsampling (`subsample` / `max_features` / `colsample_bytree`):** Submuestreo de filas y columnas para evitar el sobreajuste.

In [3]:
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd

# ==========================================
# 1. PREPARACIÓN DE DATOS
# ==========================================
# ⚠️ ¡Corregido! La columna objetivo de tu equipo se llama 'Accept'
variable_objetivo = 'Accept' 

X = df.drop(columns=[variable_objetivo]) 
y = df[variable_objetivo]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
resultados_tree_team = {}

# ==========================================
# 2. GRADIENT BOOSTING TRADICIONAL
# ==========================================
print("⏳ Entrenando Gradient Boosting (afinando depth, estimators, subsampling)...")
gb_base = GradientBoostingClassifier(random_state=42)

param_grid_gb = {
    'n_estimators': [100, 200],          
    'max_depth': [3, 5, 7],              
    'subsample': [0.8, 1.0],             
    'max_features': ['sqrt', None]       
}

gb_grid = GridSearchCV(estimator=gb_base, param_grid=param_grid_gb, scoring='f1_macro', cv=5, n_jobs=-1)
gb_grid.fit(X_train, y_train)

best_gb = gb_grid.best_estimator_
y_pred_gb = best_gb.predict(X_test)

resultados_tree_team['Gradient Boosting'] = {
    'Best Config': gb_grid.best_params_,
    'CV F1-Score': gb_grid.best_score_,
    'Test Accuracy': accuracy_score(y_test, y_pred_gb),
    'Test F1-Score': f1_score(y_test, y_pred_gb, average='macro')
}
print("✅ Gradient Boosting completado!\n")

# ==========================================
# 3. XGBOOST (EXTREME GRADIENT BOOSTING)
# ==========================================
print("⏳ Entrenando XGBoost (afinando depth, estimators, subsampling)...")
xgb_base = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

param_grid_xgb = {
    'n_estimators': [100, 200],          
    'max_depth': [3, 5, 7],              
    'subsample': [0.8, 1.0],             
    'colsample_bytree': [0.8, 1.0]       
}

xgb_grid = GridSearchCV(estimator=xgb_base, param_grid=param_grid_xgb, scoring='f1_macro', cv=5, n_jobs=-1)
xgb_grid.fit(X_train, y_train)

best_xgb = xgb_grid.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

resultados_tree_team['XGBoost'] = {
    'Best Config': xgb_grid.best_params_,
    'CV F1-Score': xgb_grid.best_score_,
    'Test Accuracy': accuracy_score(y_test, y_pred_xgb),
    'Test F1-Score': f1_score(y_test, y_pred_xgb, average='macro')
}
print("✅ XGBoost completado!\n")

# ==========================================
# 4. REPORTE FINAL PARA EL EQUIPO
# ==========================================
print("="*60)
print("🏆 REPORTE FINAL: VARIANTES DE BOOSTING (Para compartir)")
print("="*60)

for modelo, metricas in resultados_tree_team.items():
    print(f"\n🔹 {modelo}:")
    print(f"  - Mejores Configuraciones: {metricas['Best Config']}")
    print(f"  - Métrica Validación Cruzada (CV F1-Score): {metricas['CV F1-Score']:.4f}")
    print(f"  - Métrica en Prueba (Test Accuracy): {metricas['Test Accuracy']:.4f}")
    print(f"  - Métrica en Prueba (Test F1-Score): {metricas['Test F1-Score']:.4f}")
print("\n" + "="*60)

⏳ Entrenando Gradient Boosting (afinando depth, estimators, subsampling)...
✅ Gradient Boosting completado!

⏳ Entrenando XGBoost (afinando depth, estimators, subsampling)...
✅ XGBoost completado!

🏆 REPORTE FINAL: VARIANTES DE BOOSTING (Para compartir)

🔹 Gradient Boosting:
  - Mejores Configuraciones: {'max_depth': 7, 'max_features': 'sqrt', 'n_estimators': 200, 'subsample': 0.8}
  - Métrica Validación Cruzada (CV F1-Score): 0.6963
  - Métrica en Prueba (Test Accuracy): 0.8195
  - Métrica en Prueba (Test F1-Score): 0.7072

🔹 XGBoost:
  - Mejores Configuraciones: {'colsample_bytree': 1.0, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0}
  - Métrica Validación Cruzada (CV F1-Score): 0.6943
  - Métrica en Prueba (Test Accuracy): 0.8170
  - Métrica en Prueba (Test F1-Score): 0.7059



### 📌 Conclusión: Variantes de Boosting

Tras ejecutar la búsqueda de hiperparámetros (GridSearchCV) y evaluar los modelos con los datos de prueba, obtenemos las siguientes observaciones:

* **El modelo ganador:** El **Gradient Boosting** tradicional superó por un margen mínimo al XGBoost, obteniendo un **F1-Score macro de 0.7072** y un **Accuracy de 0.8195** (81.9%).
* **Estrategia del modelo ganador:** Para alcanzar este resultado, el Gradient Boosting construyó un modelo robusto (200 árboles de profundidad 7). Sin embargo, para evitar memorizar los datos (sobreajuste), la máquina determinó que era mejor entrenar cada árbol con solo el 80% de las filas (`subsample: 0.8`) y una fracción de las columnas (`max_features: 'sqrt'`).
* **Rendimiento de XGBoost:** El XGBoost obtuvo resultados casi idénticos (F1-Score de 0.7059 y Accuracy de 81.7%) pero prefirió una configuración más conservadora (100 árboles con profundidad 5 y usando el 100% de los datos).
* **Próximos pasos (Tree Team):** Estas métricas y configuraciones óptimas quedan registradas para ser comparadas contra el *Decision Tree Baseline* y las variantes de *Random Forest*. El modelo de la familia de árboles que tenga el F1-Score más alto será el que compita en la competencia final.

## RANDOM FOREST

### 1. IMPORTS + CONFIGURACIÓN

In [1]:
# === Imports generales ===
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import importlib

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

### 2. CARGAR PROYECTO + PREPROCESSING

In [2]:
# === Configurar paths ===
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# === Auto-import preprocessing modules ===
from importlib import import_module

preprocessing_modules = {}
preprocessing_dir = src_path / "preprocessing"

for module_file in sorted(preprocessing_dir.glob("*.py")):
    if module_file.stem.startswith("_"):
        continue
    module = import_module(f"preprocessing.{module_file.stem}")
    preprocessing_modules[module_file.stem] = module
    globals()[module_file.stem] = module

print("Loaded preprocessing:", ", ".join(preprocessing_modules.keys()))

Loaded preprocessing: accept, approvalDate, approvalFY, balanceGross, base_cleaning, city_bank, createJob, disbursementDate, disbursementGross, example, franchise_code, LowDoc, newExists, noemp, retainedJob, RevLineCr, urban_rural


### 3. CARGAR DATASET

In [3]:
# === Load dataset ===
df = pd.read_csv(project_root / "data" / "train.csv")

print("Shape:", df.shape)
df.head()

Shape: (20768, 21)


,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,ApprovalFY,NoEmp,...,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,1996,28,...,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,2008,1,...,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,1996,6,...,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,2011,5,...,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,2007,3,...,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0


### 4. APLICAR TODO EL PREPROCESSING

In [4]:
# === Base cleaning ===
base_cleaning = importlib.reload(base_cleaning)
df = base_cleaning.clean_base_columns(df)

# === NoEmp ===
noemp = importlib.reload(noemp)
df = noemp.preprocess_noemp(df, option="log")

# === City + Bank ===
city_bank = importlib.reload(city_bank)
df = city_bank.get_city_bank_encoder(df)

# === NewExist ===
newExists = importlib.reload(newExists)
df = newExists.preprocess_newexist(df, option="B", source_col="NewExist")

# === CreateJob ===
createJob = importlib.reload(createJob)
df = createJob.preprocess_createjob(df, option="A", source_col="CreateJob")

# === RetainedJob ===
retainedJob = importlib.reload(retainedJob)
df = retainedJob.preprocess_retainedjob(df, option="B", source_col="RetainedJob")

# === ApprovalDate ===
approvalDate = importlib.reload(approvalDate)
df = approvalDate.preprocess_approvaldate(df, option="A", source_col="ApprovalDate")

# === ApprovalFY ===
approvalFY = importlib.reload(approvalFY)
df = approvalFY.preprocess_approvalfy(df, option="B", source_col="ApprovalFY")

# === Franchise ===
franchise_code = importlib.reload(franchise_code)
df = franchise_code.preprocess_franchise_code(df, option="binary", source_col="FranchiseCode")

# === UrbanRural ===
urban_rural = importlib.reload(urban_rural)
df = urban_rural.preprocess_urban_rural(df, option="onehot", source_col="UrbanRural")

# === RevLineCr ===
RevLineCr = importlib.reload(RevLineCr)
df = RevLineCr.preprocess_revlinecr(df, option="C", source_col="RevLineCr")

# === LowDoc ===
LowDoc = importlib.reload(LowDoc)
df = LowDoc.preprocess_lowdoc(df, option="C", source_col="LowDoc")

# === DisbursementDate ===
disbursementDate = importlib.reload(disbursementDate)
df = disbursementDate.preprocess_disbursementdate(df, source_col="DisbursementDate")

# === BalanceGross ===
balanceGross = importlib.reload(balanceGross)
df = balanceGross.preprocess_balancegross(df, source_col="BalanceGross")

# === Accept ===
accept = importlib.reload(accept)
df = accept.preprocess_accept(df, source_col="Accept")

# === DisbursementGross ===
disbursementGross = importlib.reload(disbursementGross)
df = disbursementGross.preprocess_disbursementgross(df, option="B", source_col="DisbursementGross")

print("Final shape:", df.shape)

Final shape: (20765, 40)


### 5. SPLIT (COMÚN PARA TODOS)

In [5]:
# === Split ===
X = df.drop(columns=["Accept"])
y = df["Accept"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (16612, 39)
Test shape: (4153, 39)


### 6. FUNCIÓN DE EVALUACIÓN MACRO 

In [6]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(model, X_test, y_test, name="Model"):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred,average= "macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average= "macro", zero_division=0)
    roc = roc_auc_score(y_test, y_prob)

    print(f"\n=== {name} ===")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1-score (MACRO) :", round(f1, 4))
    print("ROC-AUC  :", round(roc, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_macro": f1,
        "roc_auc": roc
    }

### RANDOM FOREST BASE

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

evaluate_model(rf_model, X_test, y_test, name="Random Forest Base")


=== Random Forest Base ===
Accuracy : 0.8225
Precision: 0.7628
Recall   : 0.6828
F1-score (MACRO) : 0.707
ROC-AUC  : 0.8231

Classification Report:
              precision    recall  f1-score   support

         0.0       0.68      0.43      0.52       950
         1.0       0.85      0.94      0.89      3203

    accuracy                           0.82      4153
   macro avg       0.76      0.68      0.71      4153
weighted avg       0.81      0.82      0.81      4153



{'accuracy': 0.8225379243920058,
 'precision': 0.7627672991624981,
 'recall': 0.6828157812577025,
 'f1_macro': 0.7069849015077432,
 'roc_auc': 0.8231444205268088}

### RANDOM FOREST HIPERPARÁMETROS (TUNED)

In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Modelo base para tuning
rf = RandomForestClassifier(random_state=42)

# Grid de hiperparámetros
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "class_weight": [None, "balanced"]
}

# GridSearch
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

# Entrenamiento
grid_search.fit(X_train, y_train)

# Mejor modelo
best_rf = grid_search.best_estimator_

print("Mejores parámetros:")
print(grid_search.best_params_)

# Evaluación
tuned_results = evaluate_model(best_rf, X_test, y_test, name="Random Forest Tuned")

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Mejores parámetros:
{'class_weight': 'balanced', 'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}

=== Random Forest Tuned ===
Accuracy : 0.8086
Precision: 0.7286
Recall   : 0.7274
F1-score (MACRO) : 0.728
ROC-AUC  : 0.8249

Classification Report:
              precision    recall  f1-score   support

         0.0       0.58      0.58      0.58       950
         1.0       0.88      0.88      0.88      3203

    accuracy                           0.81      4153
   macro avg       0.73      0.73      0.73      4153
weighted avg       0.81      0.81      0.81      4153

